In [1]:
import pandas as pd

In [21]:
df = pd.read_csv('data/Superstore_tratada.csv')
df_superstore = df.copy()

In [22]:
df_superstore['Order Date'] = pd.to_datetime(df_superstore['Order Date'])
display(df_superstore.columns)
display(df_superstore.info())

Index(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
       'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code',
       'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name',
       'Sales', 'Quantity', 'Discount', 'Profit', 'unit_price',
       'sale_without_discount', 'total_cost', 'unit_cost'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Order ID               9994 non-null   object        
 1   Order Date             9994 non-null   datetime64[ns]
 2   Ship Date              9994 non-null   object        
 3   Ship Mode              9994 non-null   object        
 4   Customer ID            9994 non-null   object        
 5   Customer Name          9994 non-null   object        
 6   Segment                9994 non-null   object        
 7   Country                9994 non-null   object        
 8   City                   9994 non-null   object        
 9   State                  9994 non-null   object        
 10  Postal Code            9994 non-null   int64         
 11  Region                 9994 non-null   object        
 12  Product ID             9994 non-null   object        
 13  Cat

None

In [23]:
# Mês da primeira compra para cada cliente
primeira_compra = df_superstore.groupby('Customer ID')['Order Date'].min().dt.to_period('M')
primeira_compra.name = 'Cohort'
df_superstore = df_superstore.join(primeira_compra, on='Customer ID')

In [25]:
# Mês de cada compra
df_superstore['Buy Month'] = df_superstore['Order Date'].dt.to_period('M')

In [26]:
# Índice da cohort — quantos meses após a primeira compra
df_superstore['Cohort Index'] = (
  df_superstore['Buy Month'] - df_superstore['Cohort']
).apply(lambda x : x.n)


In [30]:
# Montando a Matriz
cohort_data = df_superstore.groupby(['Cohort', 'Cohort Index'])['Customer ID'].nunique().reset_index()
cohort_data.columns = ['Cohort', 'Index', 'Customers']

display(cohort_data)

,Cohort,Index,Customers
0,2011-01,0,31
1,2011-01,1,3
2,2011-01,3,2
3,2011-01,4,2
4,2011-01,6,2
...,...,...,...
891,2014-07,0,2
892,2014-07,5,1
893,2014-09,0,1
894,2014-10,0,2


In [ ]:
# Tamanho de cada cohort (mês 0)
cohort_size = cohort_data[cohort_data['Index'] == 0].set_index('Cohort')['Customers']

In [32]:
# Calcular percentual de retenção

cohort_data['Cohort Size'] = cohort_data['Cohort'].map(cohort_size)
cohort_data['retention %'] = (cohort_data['Customers'] / cohort_data['Cohort Size'] * 100).round(2)

In [33]:
# Exportando
cohort_data['Cohort'] = cohort_data['Cohort'].astype(str)
cohort_data.to_csv('data/cohort.csv', index=False)